# Match Prediction
Pada tahap ini model machine learning yang sudah dilatih akan digunakan untuk memprediksi hasil pertandingan antara dua negara.

# Import Library 


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import joblib

# Load Model dan Metadata

In [2]:
model_path = Path("../model/worldcup_match_model.pkl")
metadata_path = Path("../model/model_metadata.pkl")

model = joblib.load(model_path)
metadata = joblib.load(metadata_path)

print("Model berhasil dimuat dari:", model_path)
print("Metadata berhasil dimuat dari:", metadata_path)

print("Model terbaik:", metadata["best_model_name"])
print("Accuracy:", round(metadata["accuracy"], 4))

Model berhasil dimuat dari: ..\model\worldcup_match_model.pkl
Metadata berhasil dimuat dari: ..\model\model_metadata.pkl
Model terbaik: Logistic Regression
Accuracy: 0.5738


# Load Dataset Pertandingan

In [3]:
data_path = Path("../data/processed/match_processed.csv")

df = pd.read_csv(data_path)

df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)

print("Dataset berhasil dimuat dari:", data_path)
print("Jumlah baris:", df.shape[0])
print("Jumlah kolom:", df.shape[1])

df.head()

Dataset berhasil dimuat dari: ..\data\processed\match_processed.csv
Jumlah baris: 25410
Jumlah kolom: 11


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,year,result
0,2000-01-04,Egypt,Togo,2.0,1.0,Friendly,Aswan,Egypt,False,2000,home_win
1,2000-01-07,Tunisia,Togo,7.0,0.0,Friendly,Tunis,Tunisia,False,2000,home_win
2,2000-01-08,Trinidad and Tobago,Canada,0.0,0.0,Friendly,Port of Spain,Trinidad and Tobago,False,2000,draw
3,2000-01-09,Burkina Faso,Gabon,1.0,1.0,Friendly,Ouagadougou,Burkina Faso,False,2000,draw
4,2000-01-09,Guatemala,Armenia,1.0,1.0,Friendly,Los Angeles,United States,True,2000,draw


# Membuat Data Riwayat Performa Tim
Data pertandingan diubah menjadi format per tim agar bisa menghitung performa 5 pertandingan terakhir.

In [4]:
home_data = df[[
    "date", "home_team", "away_team", 
    "home_score", "away_score", "tournament", "neutral"
]].copy()

home_data.columns = [
    "date", "team", "opponent",
    "goals_for", "goals_against", "tournament", "neutral"
]

home_data["is_home"] = 1


away_data = df[[
    "date", "away_team", "home_team", 
    "away_score", "home_score", "tournament", "neutral"
]].copy()

away_data.columns = [
    "date", "team", "opponent",
    "goals_for", "goals_against", "tournament", "neutral"
]

away_data["is_home"] = 0


team_matches = pd.concat([home_data, away_data], ignore_index=True)
team_matches = team_matches.sort_values(["team", "date"]).reset_index(drop=True)

team_matches.head()

,date,team,opponent,goals_for,goals_against,tournament,neutral,is_home
0,2012-09-25,Abkhazia,Artsakh,1.0,1.0,Friendly,False,1
1,2012-10-21,Abkhazia,Artsakh,0.0,3.0,Friendly,False,0
2,2013-09-23,Abkhazia,South Ossetia,3.0,0.0,Friendly,False,1
3,2014-06-01,Abkhazia,Occitania,1.0,1.0,CONIFA World Football Cup,True,1
4,2014-06-02,Abkhazia,Sápmi,2.0,1.0,CONIFA World Football Cup,False,0


# Membuat Kolom Poin dan Hasil Tim

In [5]:
def get_team_result(row):
    if row["goals_for"] > row["goals_against"]:
        return "win"
    elif row["goals_for"] < row["goals_against"]:
        return "loss"
    else:
        return "draw"

team_matches["team_result"] = team_matches.apply(get_team_result, axis=1)

team_matches["points"] = team_matches["team_result"].map({
    "win": 3,
    "draw": 1,
    "loss": 0
})

team_matches["win"] = (team_matches["team_result"] == "win").astype(int)

team_matches.head()

,date,team,opponent,goals_for,goals_against,tournament,neutral,is_home,team_result,points,win
0,2012-09-25,Abkhazia,Artsakh,1.0,1.0,Friendly,False,1,draw,1,0
1,2012-10-21,Abkhazia,Artsakh,0.0,3.0,Friendly,False,0,loss,0,0
2,2013-09-23,Abkhazia,South Ossetia,3.0,0.0,Friendly,False,1,win,3,1
3,2014-06-01,Abkhazia,Occitania,1.0,1.0,CONIFA World Football Cup,True,1,draw,1,0
4,2014-06-02,Abkhazia,Sápmi,2.0,1.0,CONIFA World Football Cup,False,0,win,3,1


# Membuat Fungsi Mengambil Performa Terbaru Tim

In [7]:
def get_latest_team_stats(team_name, team_matches, window=5):
    team_data = team_matches[team_matches["team"] == team_name].copy()

    if team_data.empty:
        raise ValueError(f"Tim '{team_name}' tidak ditemukan di dataset.")

    latest_matches = team_data.sort_values("date").tail(window)

    stats = {
        "goals_for_last5": latest_matches["goals_for"].mean(),
        "goals_against_last5": latest_matches["goals_against"].mean(),
        "points_last5": latest_matches["points"].mean(),
        "win_rate_last5": latest_matches["win"].mean(),
        "matches_before": len(team_data)
    }

    return stats

# Membuat Fungsi Prediksi Pertandingan

In [9]:
def predict_match(home_team, away_team, tournament="FIFA World Cup", neutral=True):
    home_stats = get_latest_team_stats(home_team, team_matches)
    away_stats = get_latest_team_stats(away_team, team_matches)
    
    input_data = pd.DataFrame([{
        "home_team": home_team,
        "away_team": away_team,
        "tournament": tournament,
        "neutral": neutral,
        
        "home_goals_for_last5": home_stats["goals_for_last5"],
        "home_goals_against_last5": home_stats["goals_against_last5"],
        "home_points_last5": home_stats["points_last5"],
        "home_win_rate_last5": home_stats["win_rate_last5"],
        
        "away_goals_for_last5": away_stats["goals_for_last5"],
        "away_goals_against_last5": away_stats["goals_against_last5"],
        "away_points_last5": away_stats["points_last5"],
        "away_win_rate_last5": away_stats["win_rate_last5"],
        
        "goals_for_diff_last5": home_stats["goals_for_last5"] - away_stats["goals_for_last5"],
        "goals_against_diff_last5": home_stats["goals_against_last5"] - away_stats["goals_against_last5"],
        "points_diff_last5": home_stats["points_last5"] - away_stats["points_last5"],
        "win_rate_diff_last5": home_stats["win_rate_last5"] - away_stats["win_rate_last5"],
        "experience_diff": home_stats["matches_before"] - away_stats["matches_before"]
    }])
    
    probabilities = model.predict_proba(input_data)[0]
    classes = model.classes_
    
    result = pd.DataFrame({
        "Result": classes,
        "Probability": probabilities
    })
    
    result["Probability (%)"] = (result["Probability"] * 100).round(2)
    
    label_mapping = {
        "home_win": f"{home_team} menang",
        "draw": "Seri",
        "away_win": f"{away_team} menang"
    }
    
    result["Prediction"] = result["Result"].map(label_mapping)
    
    result = result[["Prediction", "Probability (%)"]]
    result = result.sort_values(by="Probability (%)", ascending=False).reset_index(drop=True)
    
    return result

# Uji Pertandingan

In [10]:
predict_match("Argentina", "France")

,Prediction,Probability (%)
0,Argentina menang,44.20
1,Seri,34.39
2,France menang,21.41
